# Gabarito dos Exercícios — Módulo 2
### Curso Introdutório de Python para Ciência de Dados
**Disciplina:** T326 - Ciência de Dados | UNIFOR

> ⚠️ **Atenção:** Este arquivo é o gabarito. Tente resolver os exercícios antes de consultá-lo!

---


In [13]:
# ── Setup inicial ────────────────────────────────────────────────
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns, io, warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 110, 'figure.facecolor': 'white', 'axes.grid': True, 'grid.alpha': 0.4})
sns.set_palette('Set2')

def carregar_brazilian_cities(caminho):
    with open(caminho, 'r', encoding='utf-8', newline='') as f:
        raw = f.read()
    linhas = raw.split('\r\n')
    def fix(l):
        if l.startswith('"') and l.endswith('"'): l = l[1:-1]
        return l.replace('""', '"')
    return pd.read_csv(io.StringIO('\n'.join([fix(l) for l in linhas if l.strip()])))

df = carregar_brazilian_cities('../datasets/brazilian_city.csv')
df = df.rename(columns={'IDHM Ranking 2010':'IDHM_Ranking','IBGE_CROP_PRODUCTION_$':'IBGE_CROP_PROD',
    'WAL-MART':'WALMART','IBGE_1-4':'IBGE_1a4','IBGE_5-9':'IBGE_5a9',
    'IBGE_10-14':'IBGE_10a14','IBGE_15-59':'IBGE_15a59','IBGE_60+':'IBGE_60mais'})
df['POP_FINAL'] = df['IBGE_RES_POP'].where(df['IBGE_RES_POP'] > 0, df['ESTIMATED_POP'])
df['DENSIDADE'] = np.where(df['AREA'] > 0, (df['POP_FINAL']/df['AREA']).round(2), np.nan)
df['TIPO'] = df['CAPITAL'].map({1:'Capital', 0:'Interior'})
df['IDHM_FAIXA'] = pd.cut(df['IDHM'], bins=[0,.499,.599,.699,.799,1.],
    labels=['Muito Baixo','Baixo','Médio','Alto','Muito Alto'])
regioes = {'Norte':['AM','PA','RR','RO','AC','AP','TO'],
           'Nordeste':['MA','PI','CE','RN','PB','PE','AL','SE','BA'],
           'Centro-Oeste':['MT','MS','GO','DF'],
           'Sudeste':['SP','RJ','MG','ES'],
           'Sul':['PR','SC','RS']}
df['REGIAO'] = df['STATE'].map({uf:reg for reg,ufs in regioes.items() for uf in ufs})
print("✅ Dataset carregado!")


✅ Dataset carregado!


---
## Gabarito — Módulo 2

### Exercício 1 — Municípios do CE com IDHM > 0.700

In [14]:
# ── Gabarito Exercício 2.1 ──────────────────────────────────────
ce_alto_idhm = (df[(df['STATE'] == 'CE') & (df['IDHM'] > 0.700)]
                [['CITY', 'IDHM', 'GDP_CAPITA']]
                .sort_values('IDHM', ascending=False)
                .reset_index(drop=True))

ce_alto_idhm.index += 1
print(f"Municípios do CE com IDHM > 0.700: {len(ce_alto_idhm)}")
print(ce_alto_idhm.to_string())


Municípios do CE com IDHM > 0.700: 4
        CITY   IDHM  GDP_CAPITA
1  Fortaleza  0.754     23045.0
2     Sobral  0.714     20258.0
3      Crato  0.713     11642.0
4    Eusébio  0.701     59098.0


### Exercício 2 — Agrupamento por Estado

In [15]:
# ── Gabarito Exercício 2.2 ──────────────────────────────────────
por_estado = df.groupby('STATE').agg(
    Pop_Total=('POP_FINAL', 'sum'),
    IDHM_Medio=('IDHM', 'mean')
).round(3).sort_values('Pop_Total', ascending=False)

print("Top 5 estados mais populosos:")
print(por_estado.head(5).to_string())


Top 5 estados mais populosos:
       Pop_Total  IDHM_Medio
STATE                       
SP      41357343       0.740
MG      19597330       0.668
RJ      16027462       0.709
BA      14027251       0.592
RS      10696897       0.709


### Exercício 3 — Frota Total e por Habitante

In [16]:
# ── Gabarito Exercício 2.3 ──────────────────────────────────────
df['FROTA_TOTAL'] = df['Cars'] + df['Motorcycles']

frota_estado = df.groupby('STATE').agg(
    Pop_Total=('POP_FINAL', 'sum'),
    Frota_Total=('FROTA_TOTAL', 'sum')
)
frota_estado['Frota_por_Hab'] = (frota_estado['Frota_Total'] / frota_estado['Pop_Total']).round(4)
frota_estado = frota_estado.sort_values('Frota_por_Hab', ascending=False)

print("Frota por habitante — Top 10 estados:")
print(frota_estado.head(10).to_string())


Frota por habitante — Top 10 estados:
       Pop_Total  Frota_Total  Frota_por_Hab
STATE                                       
SC       6271028      4094167         0.6529
DF       2570160      1499499         0.5834
SP      41357343     23956157         0.5792
PR      10452673      5910903         0.5655
RS      10696897      5600673         0.5236
MT       3035122      1583793         0.5218
RO       1562409       803757         0.5144
GO       6003788      3025623         0.5040
MS       2454479      1221043         0.4975
MG      19597330      9050944         0.4618


### Exercício 4 — Capitais vs Interior

In [17]:
# ── Gabarito Exercício 2.4 ──────────────────────────────────────
pib_tipo = df.groupby('TIPO')['GDP_CAPITA'].mean()
print("PIB per Capita médio:")
print(pib_tipo.round(2))

dif_pct = ((pib_tipo['Capital'] - pib_tipo['Interior']) / pib_tipo['Interior'] * 100)
print(f"\nCapitais têm PIB per capita {dif_pct:.1f}% {'maior' if dif_pct>0 else 'menor'} que o interior")


PIB per Capita médio:
TIPO
Capital     30009.52
Interior    21057.87
Name: GDP_CAPITA, dtype: float64

Capitais têm PIB per capita 42.5% maior que o interior


### Exercício 5 — IDHM por Classificação Urbana/Rural

In [18]:
# ── Gabarito Exercício 2.5 ──────────────────────────────────────
stats_rural_urbano = df.groupby('RURAL_URBAN')['IDHM'].describe().round(3)
print("Estatísticas do IDHM por Classificação:")
print(stats_rural_urbano.to_string())


Estatísticas do IDHM por Classificação:
                          count   mean    std    min    25%    50%    75%    max
RURAL_URBAN                                                                     
0                           5.0  0.228  0.313  0.000  0.000  0.000  0.555  0.587
Intermediário Adjacente   688.0  0.664  0.061  0.505  0.613  0.674  0.713  0.854
Intermediário Remoto       60.0  0.621  0.067  0.483  0.563  0.636  0.673  0.742
Rural Adjacente          3040.0  0.640  0.068  0.000  0.585  0.639  0.696  0.806
Rural Remoto              323.0  0.598  0.063  0.418  0.558  0.593  0.650  0.788
Sem classificação           5.0  0.000  0.000  0.000  0.000  0.000  0.000  0.000
Urbano                   1457.0  0.711  0.059  0.503  0.678  0.724  0.753  0.862
